# Garbage Classification Image Collector

YOLO 学習用にゴミ分別画像を収集する Colab ノートブック。

**カテゴリ**
1. `plastic_container` — プラ容器 (カップ・トレイ・袋)
2. `paper_cardboard` — 紙・段ボール
3. `pet_bottle` — ペットボトル

**フロー**
1. `icrawler` で Bing / Google から各キーワードを検索 → `_raw/` に保存 (ハッシュ命名で自動重複除外)
2. CLIP (ViT-B/32) で「正解カテゴリ / 他カテゴリ / その他 (ノイズ)」の4値ゼロショット分類 → `scores.csv`
3. 正解が最高スコアのものから上位 N 枚を `{category}_001.jpg` 形式で各カテゴリフォルダに保存

**再開可能性**
- 各カテゴリの収集セルは独立実行可
- 既存の `_raw/` ファイルはハッシュで重複検出するので再実行しても増えるだけ
- `scores.csv` を作り直せば選別だけやり直せる (再DL不要)

**所要時間の目安**
- 収集: 1カテゴリあたり 3〜8 分 (キーワード数 × エンジン数 × ネットワーク次第、tqdm で ETA 表示)
- CLIP 採点: 全体で 30 秒〜2 分 (GPUなら高速)

**Drive 出力**
```
MyDrive/garbage_dataset/
├── plastic_container/       N枚 (FINAL_PER_CATEGORY)
├── paper_cardboard/         N枚
├── pet_bottle/              N枚
├── _raw/{category}/         CLIP 前の全収集画像
└── scores.csv               CLIP スコア
```


## 1. セットアップ

In [ ]:
# [v2追加バッチ] 実行: ランタイム再起動した場合のみ必須 (既インストール済みなら不要)
!pip install -q icrawler transformers


In [ ]:
# [v2追加バッチ] 実行: 既に Drive マウント中ならスキップ可
from google.colab import drive
drive.mount('/content/drive')


### 1-1. パラメータ設定 (このセルを編集して挙動を変えられます)

**よく変えるもの**
- `PER_KEYWORD`: 1キーワードで何枚集めるか (多いほど時間↑、母数↑)
- `FINAL_PER_CATEGORY`: 最終的に採用する枚数 (50 → 100 などに変更可)
- `ENGINES`: 使う検索エンジン (`('bing', 'google')` / `('bing',)` など)
- `CATEGORIES[...]['keywords']`: 検索クエリ
- `CATEGORIES[...]['clip_prompt']`: CLIP でその category を説明するプロンプト
- `OTHER_PROMPT`: ノイズ判定用のプロンプト

**たまに変えるもの**
- `MIN_IMAGE_SIZE`: 小さすぎる画像を捨てる閾値
- `CLIP_BATCH_SIZE`: CLIP 採点時のバッチサイズ (VRAM 次第)
- `CLIP_MODEL`: 精度↑が欲しいなら `openai/clip-vit-large-patch14` (遅く・重くなる)
- `DRIVE_ROOT`: 出力先


In [ ]:
# [v2追加バッチ] 実行必須 — パラメータを現在値(PER_KEYWORD=50, 英語キーワード追加)で反映させる
from pathlib import Path

# ---- Drive 出力先 ----
DRIVE_ROOT = Path('/content/drive/MyDrive/garbage_dataset')
RAW_DIR = DRIVE_ROOT / '_raw'
SCORES_CSV = DRIVE_ROOT / 'scores.csv'

# ---- カテゴリ定義 (キーワード・CLIPプロンプト) ----
# 母数を増やすため英語キーワードを追加 (Bing は日英どちらも返す)
CATEGORIES = {
    'plastic_container': {
        'keywords': [
            'プラスチック容器 ゴミ',
            '使用済み プラカップ',
            '食品トレイ 発泡スチロール',
            'レジ袋 プラスチック',
            'プラごみ 分別',
            'plastic waste trash',
            'used plastic cups',
            'plastic food tray waste',
            'plastic bag garbage',
        ],
        'clip_prompt': 'a photo of a used plastic container, cup, tray, or plastic bag',
    },
    'paper_cardboard': {
        'keywords': [
            '段ボール ゴミ',
            '古紙 回収',
            '新聞紙 束',
            '紙くず ゴミ',
            'ダンボール箱 つぶす',
            'cardboard box waste',
            'waste paper recycling',
            'newspaper bundle recycling',
            'flattened cardboard trash',
        ],
        'clip_prompt': 'a photo of waste paper, newspapers, or cardboard boxes',
    },
    'pet_bottle': {
        'keywords': [
            'ペットボトル ゴミ',
            '空のペットボトル',
            'ペットボトル 回収',
            'PETボトル 分別',
            'ペットボトル つぶす',
            'empty PET bottle',
            'plastic bottle recycling',
            'crushed PET bottle',
            'discarded plastic bottle',
        ],
        'clip_prompt': 'a photo of an empty PET plastic bottle',
    },
}

OTHER_PROMPT = 'a photo of something else, an illustration, a logo, or a person'

# ---- 収集・選別の枚数 ----
PER_KEYWORD = 50                 # 1キーワードあたりの取得上限 (母数不足対策で 25 → 50)
FINAL_PER_CATEGORY = 50          # 最終採用枚数 / カテゴリ
MIN_IMAGE_SIZE = (200, 200)      # これ未満は破棄

# ---- v2 (追加バッチ) 設定 ----
FINAL_PER_CATEGORY_V2 = 100              # 追加バッチで採用する枚数
V2_START_INDEX = FINAL_PER_CATEGORY + 1  # 追加バッチのファイル名開始番号 (= 51)
V2_SUFFIX = '_v2'                        # 追加バッチ出力フォルダの接尾辞

# ---- 検索エンジン ----
ENGINES = ('bing',)

# ---- CLIP 設定 ----
CLIP_MODEL = 'openai/clip-vit-base-patch32'
CLIP_BATCH_SIZE = 32

CATEGORY_ORDER = list(CATEGORIES.keys())

for c in CATEGORY_ORDER:
    (RAW_DIR / c).mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / c).mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / f'{c}{V2_SUFFIX}').mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Categories:', CATEGORY_ORDER)
print('Engines:', ENGINES)
print('Per keyword:', PER_KEYWORD, '| Final:', FINAL_PER_CATEGORY,
      '| +v2:', FINAL_PER_CATEGORY_V2, f'(->{V2_SUFFIX}/)')
print('Keywords per cat:', {c: len(CATEGORIES[c]['keywords']) for c in CATEGORY_ORDER})


## 2. 画像収集 (icrawler)

各カテゴリのセルは**独立して実行可能**。途中で Colab が切れても、もう一度同じセルを実行すれば既存ハッシュをスキップして続きから収集します。

`_raw/{category}/` には `{hash}.jpg` 形式で保存 (重複除外用)。最終リネームは CLIP 選別後に行います。

**進捗表示**: tqdm で `(キーワード×エンジン)` の消化を `3/10 [01:23<03:45]` のように表示。ETA が出ます。


In [ ]:
# [v2追加バッチ] 実行必須 — collect_category 関数を定義する
import hashlib
import logging
import shutil
from tqdm.auto import tqdm
from icrawler.builtin import GoogleImageCrawler, BingImageCrawler

# icrawler はデフォルトで大量のログを出して進捗表示と干渉するため ERROR のみに抑制
for name in ('icrawler', 'icrawler.crawler', 'icrawler.downloader',
             'icrawler.parser', 'icrawler.feeder'):
    logging.getLogger(name).setLevel(logging.ERROR)

def file_hash(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def collect_category(category, engines=None):
    engines = tuple(engines) if engines is not None else ENGINES
    cat_raw = RAW_DIR / category
    cat_raw.mkdir(parents=True, exist_ok=True)

    keywords = CATEGORIES[category]['keywords']
    before = len(list(cat_raw.glob('*.*')))

    print(f'\n========== 画像収集開始: [{category}] ==========')
    print(f'  既存画像数 : {before} 枚 (_raw/{category}/)')
    print(f'  検索エンジン: {list(engines)}')
    print(f'  キーワード ({len(keywords)} 件):')
    for kw in keywords:
        print(f'    ・{kw}')
    print(f'  1キーワードあたり最大: {PER_KEYWORD} 枚  /  最小画像サイズ: {MIN_IMAGE_SIZE}')
    print('-' * 50)

    tmp_root = Path(f'/content/tmp_{category}')
    if tmp_root.exists():
        shutil.rmtree(tmp_root)

    jobs = [(i, kw, eng) for i, kw in enumerate(keywords) for eng in engines]
    total_added = 0
    total_dup = 0
    total_failed = 0

    with tqdm(total=len(jobs), desc=f'[{category}] 収集中', unit='query') as pbar:
        for i, kw, engine in jobs:
            pbar.set_postfix_str(f'{engine} / {kw}')
            tmp_dir = tmp_root / f'{i:02d}_{engine}'
            tmp_dir.mkdir(parents=True, exist_ok=True)

            crawler_cls = GoogleImageCrawler if engine == 'google' else BingImageCrawler
            crawler = crawler_cls(
                storage={'root_dir': str(tmp_dir)},
                feeder_threads=1,
                parser_threads=1,
                downloader_threads=4,
                log_level=logging.ERROR,
            )
            try:
                crawler.crawl(
                    keyword=kw,
                    max_num=PER_KEYWORD,
                    min_size=MIN_IMAGE_SIZE,
                    file_idx_offset=0,
                )
            except Exception as e:
                pbar.write(f'  [失敗] {engine:6s} "{kw}"  →  {type(e).__name__}: {e}')
                total_failed += 1
                pbar.update(1)
                continue

            added = 0
            dup = 0
            for src in tmp_dir.glob('*.*'):
                if src.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                    continue
                try:
                    h = file_hash(src)
                except Exception:
                    continue
                ext = '.jpg' if src.suffix.lower() == '.jpeg' else src.suffix.lower()
                dst = cat_raw / f'{h[:16]}{ext}'
                if dst.exists():
                    dup += 1
                else:
                    shutil.copy2(src, dst)
                    added += 1
            total_added += added
            total_dup += dup
            pbar.write(f'  {engine:6s} "{kw}"  →  新規 {added:3d} 枚 / 重複スキップ {dup:3d} 枚')
            pbar.update(1)

    shutil.rmtree(tmp_root, ignore_errors=True)
    after = len(list(cat_raw.glob('*.*')))

    print('-' * 50)
    print(f'========== 収集完了: [{category}] ==========')
    print(f'  新規追加  : +{total_added} 枚')
    print(f'  重複スキップ: {total_dup} 枚  /  失敗クエリ: {total_failed} 件')
    print(f'  _raw 合計 : {after} 枚  (収集前 {before} → 収集後 {after})')
    if after < FINAL_PER_CATEGORY * 2:
        print(f'  ⚠ 最終採用 {FINAL_PER_CATEGORY} 枚に対して母数が少なめです。'
              f'もう一度実行するか PER_KEYWORD を増やすと安定します。')


### 2-1. プラ容器

In [ ]:
# [v2追加バッチ] 実行必須 — _raw/plastic_container/ に追加クロール (ハッシュ重複は自動スキップ)
collect_category('plastic_container')


### 2-2. 紙・段ボール

In [ ]:
# [v2追加バッチ] 実行必須 — _raw/paper_cardboard/ に追加クロール
collect_category('paper_cardboard')


### 2-3. ペットボトル

In [ ]:
# [v2追加バッチ] 実行必須 — _raw/pet_bottle/ に追加クロール
collect_category('pet_bottle')


## 3. CLIP ゼロショット分類

各画像を 4 つのプロンプトとの類似度で評価し、正解カテゴリが最高スコアのものだけ採用します。

tqdm でカテゴリごとの進捗を表示 (GPU なら数十秒で終わります)。


In [ ]:
# [v2追加バッチ] 実行必須 — CLIP モデル/プロセッサをロード
import torch
from transformers import CLIPModel, CLIPProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| model:', CLIP_MODEL)

clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)

PROMPTS = [CATEGORIES[c]['clip_prompt'] for c in CATEGORY_ORDER] + [OTHER_PROMPT]
for p in PROMPTS:
    print('  -', p)


In [ ]:
# [v2追加バッチ] 実行必須 — 追加クロール分を含めて scores.csv を再生成
import csv
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

# transformers の版差で get_*_features の戻り値 (tensor / ModelOutput の中身) が
# ブレるため、サブモデルを直接呼んで pooler_output に projection をかける。
@torch.no_grad()
def encode_text(prompts):
    inputs = clip_processor(text=prompts, return_tensors='pt', padding=True).to(device)
    out = clip_model.text_model(**inputs)
    feats = clip_model.text_projection(out.pooler_output)
    return feats / feats.norm(dim=-1, keepdim=True)

@torch.no_grad()
def encode_images(pil_images):
    inputs = clip_processor(images=pil_images, return_tensors='pt').to(device)
    out = clip_model.vision_model(**inputs)
    feats = clip_model.visual_projection(out.pooler_output)
    return feats / feats.norm(dim=-1, keepdim=True)

def load_image(p):
    try:
        img = Image.open(p).convert('RGB')
        if min(img.size) < MIN_IMAGE_SIZE[0]:
            return None
        return img
    except (UnidentifiedImageError, OSError):
        return None

print('\n========== CLIP スコアリング開始 ==========')
text_feats = encode_text(PROMPTS)
print(f'  テキスト特徴量: {tuple(text_feats.shape)}  (プロンプト {len(PROMPTS)} 件)')

rows = []
for cat in CATEGORY_ORDER:
    cat_raw = RAW_DIR / cat
    paths = sorted(p for p in cat_raw.glob('*.*') if p.is_file())
    print(f'\n----- [{cat}] -----')
    print(f'  対象画像: {len(paths)} 枚')
    if not paths:
        print(f'  画像がありません — スキップ (先に collect_category({cat!r}) を実行)')
        continue

    n_ok = 0
    n_skip = 0
    pbar = tqdm(range(0, len(paths), CLIP_BATCH_SIZE),
                desc=f'[{cat}] CLIP採点', unit='batch')
    for i in pbar:
        chunk = paths[i:i + CLIP_BATCH_SIZE]
        imgs, kept = [], []
        for p in chunk:
            img = load_image(p)
            if img is not None:
                imgs.append(img)
                kept.append(p)
            else:
                n_skip += 1
        if not imgs:
            pbar.set_postfix_str(f'OK {n_ok} / 読込失敗 {n_skip}')
            continue
        img_feats = encode_images(imgs)
        sims = (img_feats @ text_feats.T).cpu().numpy()
        for p, s in zip(kept, sims):
            rows.append([cat, p.name,
                         float(s[0]), float(s[1]), float(s[2]), float(s[3])])
            n_ok += 1
        pbar.set_postfix_str(f'OK {n_ok} / 読込失敗 {n_skip}')
    print(f'  採点成功: {n_ok} 枚  /  読込不能でスキップ: {n_skip} 枚')

with open(SCORES_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['raw_category', 'filename',
                'score_plastic', 'score_paper', 'score_pet', 'score_other'])
    w.writerows(rows)

print(f'\n========== CLIP スコアリング完了 ==========')
print(f'  保存先   : {SCORES_CSV}')
print(f'  採点合計 : {len(rows)} 枚')


## 4. 上位 N 枚を選別して最終フォルダへ

`scores.csv` から各カテゴリの上位 `FINAL_PER_CATEGORY` 枚を選んで `{category}/{category}_NNN.jpg` に連番コピー。

**この手順は何度でもやり直せます** — `_raw/` と `scores.csv` を保持しているので、枚数・プロンプト・閾値を変えたければこのセルだけ再実行すれば OK。


In [ ]:
# [v2追加バッチ] 実行: スキップ推奨 — 既存 {category}/ (v1 50枚) を保持したい場合は動かさない
# (実行すると v1 50枚が最新 _raw の top-50 で上書きされる。追加クロールの結果で v1 顔ぶれが入れ替わる可能性あり)
import shutil
import pandas as pd

df = pd.read_csv(SCORES_CSV)
score_cols = ['score_plastic', 'score_paper', 'score_pet', 'score_other']
col_to_cat = {
    'score_plastic': 'plastic_container',
    'score_paper': 'paper_cardboard',
    'score_pet': 'pet_bottle',
    'score_other': 'other',
}
df['pred'] = df[score_cols].idxmax(axis=1).map(col_to_cat)

cat_to_col = {
    'plastic_container': 'score_plastic',
    'paper_cardboard': 'score_paper',
    'pet_bottle': 'score_pet',
}

print(f'\n========== 最終選別開始 ==========')
print(f'  scores.csv 読み込み: {len(df)} 行')
print(f'  目標枚数 / カテゴリ: {FINAL_PER_CATEGORY} 枚')

summary = []
for cat, col in cat_to_col.items():
    final_dir = DRIVE_ROOT / cat

    removed = 0
    for p in final_dir.glob('*.*'):
        p.unlink()
        removed += 1

    raw_count = int((df['raw_category'] == cat).sum())
    hits = df[(df['raw_category'] == cat) & (df['pred'] == cat)]
    sub = hits.sort_values(col, ascending=False).head(FINAL_PER_CATEGORY)

    print(f'\n----- [{cat}] -----')
    print(f'  既存ファイル削除: {removed} 枚')
    print(f'  _raw 画像       : {raw_count} 枚')
    print(f'  CLIP で正解判定 : {len(hits)} 枚 ({len(hits)/max(raw_count,1)*100:.1f}%)')
    print(f'  採用             : {len(sub)} / {FINAL_PER_CATEGORY} 枚')
    if len(sub):
        print(f'  採用スコア範囲  : {float(sub[col].min()):.3f} 〜 {float(sub[col].max()):.3f}')

    for i, row in enumerate(sub.itertuples(index=False), start=1):
        src = RAW_DIR / cat / row.filename
        ext = Path(row.filename).suffix.lower()
        dst = final_dir / f'{cat}_{i:03d}{ext}'
        shutil.copy2(src, dst)

    status = 'OK' if len(sub) >= FINAL_PER_CATEGORY else f'不足 {FINAL_PER_CATEGORY - len(sub)}'
    summary.append((cat, raw_count, len(hits), len(sub), status))
    if len(sub) < FINAL_PER_CATEGORY:
        print(f'  ⚠ 目標未達 — collect_category({cat!r}) を再実行して母数を増やしてください')

print(f'\n========== 最終選別完了 ==========')
print(f'  {"category":<20} {"_raw":>6} {"CLIP通過":>10} {"採用":>6}  状態')
for cat, raw_count, hit_count, adopted, status in summary:
    print(f'  {cat:<20} {raw_count:>6} {hit_count:>10} {adopted:>6}  {status}')


## 4-2. 追加バッチ (v2) を別フォルダに出力

既存の `{category}/` 50枚は保持したまま、**追加で `FINAL_PER_CATEGORY_V2` 枚** を `{category}_v2/` に出力します。
ファイル名は `{category}_{V2_START_INDEX:03d}..` (デフォルトだと `_051` 〜 `_150`) と連番で、v1 と結合しても番号衝突しません。

**重複除外**: 既存 `{category}/` にあるファイルを内容ハッシュ (md5 先頭16文字) で検出し、それらを候補から外したうえで CLIP スコア上位 `FINAL_PER_CATEGORY_V2` 枚を採用します。

**前提**
- `collect_category(...)` を追加クロールして `_raw/` の母数を増やしてある
- セクション 3 (CLIP 採点) を再実行して `scores.csv` を最新化してある

母数が不足 (= `_raw` の CLIP 通過枚数が既存50 + 追加100 に届かない) 場合は、`PER_KEYWORD` を増やすか `ENGINES = ('bing', 'google')` に拡張して収集セルを再実行してください。


In [ ]:
# [v2追加バッチ] ★実行必須 ★ — 追加100枚を {category}_v2/_051..150 に出力する本体セル
import hashlib
import shutil
import pandas as pd
from pathlib import Path

def _content_hash16(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()[:16]

df = pd.read_csv(SCORES_CSV)
score_cols = ['score_plastic', 'score_paper', 'score_pet', 'score_other']
col_to_cat = {
    'score_plastic': 'plastic_container',
    'score_paper': 'paper_cardboard',
    'score_pet': 'pet_bottle',
    'score_other': 'other',
}
df['pred'] = df[score_cols].idxmax(axis=1).map(col_to_cat)

cat_to_col = {
    'plastic_container': 'score_plastic',
    'paper_cardboard': 'score_paper',
    'pet_bottle': 'score_pet',
}

print('\n========== 追加バッチ (v2) 選別開始 ==========')
print(f'  scores.csv 読み込み   : {len(df)} 行')
print(f'  既存枚数 / カテゴリ   : {FINAL_PER_CATEGORY} 枚 ({{category}}/)')
print(f'  追加目標 / カテゴリ   : {FINAL_PER_CATEGORY_V2} 枚 ({{category}}{V2_SUFFIX}/)')
print(f'  ファイル名番号         : _{V2_START_INDEX:03d} 〜 _{V2_START_INDEX + FINAL_PER_CATEGORY_V2 - 1:03d}')

summary_v2 = []
for cat, col in cat_to_col.items():
    v1_dir = DRIVE_ROOT / cat
    v2_dir = DRIVE_ROOT / f'{cat}{V2_SUFFIX}'
    v2_dir.mkdir(parents=True, exist_ok=True)

    # 既存 v1 (= {cat}/*) の内容ハッシュ集合。_raw のファイル名 stem と同じ hash16 になる。
    existing_hashes = {_content_hash16(p) for p in v1_dir.glob('*.*') if p.is_file()}

    # v2 は毎回クリアして作り直す (再実行可)。v1 には触れない。
    removed = 0
    for p in v2_dir.glob('*.*'):
        p.unlink()
        removed += 1

    hits = df[(df['raw_category'] == cat) & (df['pred'] == cat)].copy()
    # _raw のファイル名は '{hash16}.ext' 形式 (cell 7 で命名) なので stem == hash16
    hits['hash16'] = hits['filename'].map(lambda fn: Path(fn).stem)
    fresh = hits[~hits['hash16'].isin(existing_hashes)]
    sub = fresh.sort_values(col, ascending=False).head(FINAL_PER_CATEGORY_V2)

    print(f'\n----- [{cat}] -----')
    print(f'  v2 既存ファイル削除  : {removed} 枚')
    print(f'  CLIP で正解判定      : {len(hits)} 枚')
    print(f'  既存 v1 と重複除外   : {len(hits) - len(fresh)} 枚')
    print(f'  採用候補 (新規)      : {len(fresh)} 枚')
    print(f'  採用                 : {len(sub)} / {FINAL_PER_CATEGORY_V2} 枚')
    if len(sub):
        print(f'  採用スコア範囲       : {float(sub[col].min()):.3f} 〜 {float(sub[col].max()):.3f}')

    for i, row in enumerate(sub.itertuples(index=False), start=V2_START_INDEX):
        src = RAW_DIR / cat / row.filename
        ext = Path(row.filename).suffix.lower()
        dst = v2_dir / f'{cat}_{i:03d}{ext}'
        shutil.copy2(src, dst)

    status = 'OK' if len(sub) >= FINAL_PER_CATEGORY_V2 else f'不足 {FINAL_PER_CATEGORY_V2 - len(sub)}'
    summary_v2.append((cat, len(hits), len(fresh), len(sub), status))
    if len(sub) < FINAL_PER_CATEGORY_V2:
        print(f'  ⚠ 目標未達 — collect_category({cat!r}) を再実行するか PER_KEYWORD を増やしてください')

print(f'\n========== 追加バッチ (v2) 選別完了 ==========')
print(f'  {"category":<20} {"CLIP通過":>10} {"新規候補":>10} {"採用":>6}  状態')
for cat, hit_count, fresh_count, adopted, status in summary_v2:
    print(f'  {cat:<20} {hit_count:>10} {fresh_count:>10} {adopted:>6}  {status}')


## 5. 結果プレビュー

In [ ]:
# [v2追加バッチ] 実行: 任意 — 結果を目視したいときだけ (v1 と v2 を順に表示)
import math
import matplotlib.pyplot as plt
from PIL import Image

def show_grid(folder_name, n_cols=10):
    final_dir = DRIVE_ROOT / folder_name
    paths = sorted(final_dir.glob('*.*'))
    if not paths:
        print(f'[{folder_name}] no images')
        return
    n_rows = math.ceil(len(paths) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.6, n_rows * 1.6))
    fig.suptitle(f'{folder_name}  ({len(paths)} images)', fontsize=14)
    axes_flat = axes.flat if hasattr(axes, 'flat') else [axes]
    for ax, p in zip(axes_flat, paths):
        try:
            ax.imshow(Image.open(p))
        except Exception:
            pass
        ax.axis('off')
        ax.set_title(p.stem.split('_')[-1], fontsize=6)
    for ax in list(axes_flat)[len(paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

for cat in CATEGORY_ORDER:
    show_grid(cat)
    show_grid(f'{cat}{V2_SUFFIX}')


## 6. トラブルシュート

### 50 枚に届かない場合
該当カテゴリの `collect_category(...)` セルをもう一度実行 → その後セクション 3, 4 を再実行。
`PER_KEYWORD` を 40 などに増やしたり、`CATEGORIES[cat]['keywords']` にキーワードを追加する手もあります。

### ノイズが多い/精度を上げたい
- `CATEGORIES[...]['clip_prompt']` を編集して再実行 (セクション 3, 4 のみ)
- `OTHER_PROMPT` を強化 (例: `'a cartoon illustration or a product advertisement'`)
- `CLIP_MODEL = 'openai/clip-vit-large-patch14'` に切替 (VRAM 次第で重い)

### Colab がタイムアウトした
どのセルでも、もう一度実行すれば既存ファイル/スコアを活かして続きから再開します。
